# 04. Scenario 3 — "I want to verify this overseas approval alert" (risk verification sub-flow)

The third support scenario. The tool succeeds, but its result carries a **fraud/AML monitoring flag**. When D7 (risk check) detects it, the case passes through the conditional sub-flow **run the KYC re-verification tool -> D8 (pass/fail decision)** before closing.

In the second half you will probe the **guardrail (D1) and its calibrated confidence** — we rephrase the same situation as a victim report, run it through both backends, and watch how the labels and confidences respond.

In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
graphs, client = W.make_graphs()
sc = W.scenario(3)
print("Scenario:", sc["title"])
print("Customer question:", sc["question"])
print("script (pre-set for reproducibility):", sc["script"])   # risk_flag: True, kyc_result: verified


## Where this scenario sits in the full workflow

The path this scenario takes (red borders): the successful tool result carries a monitoring flag, so the case passes through the **risk_check -> kyc_verify (D8)** sub-flow before closing. This is the financial-industry rule "re-verify identity before closing a high-risk transaction", expressed in the graph.

Assigned team (red box): **card_security (lost/stolen/security)** — a suspicious-charge verification context. The model sometimes picks cards instead; compare with the D2 decision in the STEP output.
> Note: this diagram is not auto-generated from the running LangGraph; it is a **visualization-only drawing** rendered from the caption dictionaries in `ws_common.py`, keyed by node name (details: `[EN]_01_build_agent.ipynb`).


In [ ]:
W.draw_graph(highlight=W.scenario_path(3), show_teams=True, width=760)

## (1) Run with Haiku 4.5 decisions

In the STEP output, confirm that after risk_check decides `needs_verification`, the **kyc_verify node runs as an extra step**.


In [ ]:
res_haiku = W.run_verbose(graphs["Haiku"], sc["question"], sc["script"], "Haiku")

## (2) Run with strands-decider (E4B) decisions


In [ ]:
res_strands = W.run_verbose(graphs["StrandsDecider"], sc["question"], sc["script"], "StrandsDecider")

## (3) Execution time comparison


In [ ]:
W.compare(res_haiku, res_strands)

## (4) Teaching moment: guardrail sensitivity — victim phrasing on both backends

The scenario question above was deliberately written in neutral wording. Now we rephrase the same situation as a **victim report** — the way a customer would actually say it ("I just got an alert for an overseas payment I did not make... please block my card") — and run it through both arms unchanged.

The safety decision (D1) is the hardest call for a small decision model — a victim *report* and a compliance *violation* share much of their surface vocabulary. Generic small models used zero-shot often over-block such phrasings (misreading them as violations). Watch how strands-decider labels this phrasing, and **how its confidence shifts relative to the neutral wording**.

In [ ]:
victim_q = ("I just got an alert for an overseas payment I did not make. LUXE GADGETS, "
            "USD 1,284, card ending 7712. Please block my card.")

res_haiku_v = W.run_verbose(graphs["Haiku"], victim_q, sc["script"], "Haiku (victim phrasing)")


In [ ]:
res_strands_v = W.run_verbose(graphs["StrandsDecider"], victim_q, sc["script"], "StrandsDecider (victim phrasing)")

### Compare D1 alone across phrasings

Four phrasings (three charged victim reports + the neutral one) go to D1 only, with both backends side by side. Watch the labels, and watch **whether the confidence tracks how charged/ambiguous each phrasing is** — on emotionally charged report phrasings it should come back lower than on the neutral one. Only a **calibrated** confidence behaves this way, and only then can you operate thresholds on it.

In [ ]:
from IPython.display import Markdown, display
from agent.prompts import D1_QUESTION
from agent.state import D1_CHOICES
from agent.decisions import SystemOneDecision, HaikuDecision
from common import BEDROCK_MODEL_IDS, PRICING

sd = SystemOneDecision()
hk = HaikuDecision(client, BEDROCK_MODEL_IDS["haiku-4-5"], PRICING["haiku-4-5"])
probes = [
    victim_q,
    "There's an overseas charge approval alert I never made. Block it right now!",
    "There is a transaction on my card that I don't recognize. Can you check it?",
    sc["question"],   # the neutral phrasing used by this scenario
]
rows = ["| phrase | StrandsDecider D1 (confidence) | Haiku D1 |", "|---|---|---|"]
for t in probes:
    r_sd = sd.decide(state_text=t, question=D1_QUESTION, choices=D1_CHOICES, decision_id="d1")
    r_hk = hk.decide(state_text=t, question=D1_QUESTION, choices=D1_CHOICES, decision_id="d1")
    rows.append(f"| {t[:60]}... | {r_sd.label} ({r_sd.confidence:.3f}) | {r_hk.label} |")
display(Markdown("\n".join(rows)))

**What to look for**
- The dedicated decision model (strands-decider E4B) labels the victim phrasings **in_scope, correctly** — the over-blocking that generic small zero-shot models often show on such phrasings does not reproduce, in English or in Korean ([KR] notebook). That is the benefit of a model *adapted* for decision tasks.
- More important is the **behavior of the confidence**: on charged report phrasings it comes back visibly lower than on the neutral wording. Because the confidence is calibrated, a **cascade** — "escalate D1 decisions below a confidence threshold to a stronger model (an LLM)" — actually works.
- The production design principle stands: **give the safety decision (D1) a threshold-based escalation path**, and let the decision layer own routing/review decisions (D2-D8). A calibrated model is what makes this setup safe.
- Next: `[EN]_05_bedrock_intelligent_routing.ipynb` — a managed router that picks *which generation model* a call goes to, rather than making decisions.
